# Logprob Metric Distributions

Visualize scalar logprob metrics for each dataset/model experiment and HG/MG split. The notebook reads existing metric outputs from `TEXT_DETECTION_DATA/processed/{dataset}__{model}/{split}/metrics/metrics.parquet`.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)

## Load Metrics

Set `TEXT_DETECTION_DATA` before opening the notebook, or edit `DATA_ROOT` below for local exploration.

In [ ]:
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"

def parse_experiment_name(experiment_name: str) -> tuple[str, str]:
    if "__" not in experiment_name:
        return experiment_name, "unknown"
    dataset, model = experiment_name.split("__", maxsplit=1)
    return dataset, model

def discover_metric_files(processed_root: Path) -> list[Path]:
    return sorted(processed_root.glob("*/*/metrics/metrics.parquet"))

def load_metrics(processed_root: Path) -> pd.DataFrame:
    frames = []
    for path in discover_metric_files(processed_root):
        experiment_name = path.parents[2].name
        split = path.parents[1].name
        dataset, model = parse_experiment_name(experiment_name)
        frame = pd.read_parquet(path)
        frame.insert(0, "dataset", dataset)
        frame.insert(1, "model", model)
        frame.insert(2, "experiment_name", experiment_name)
        frame["split"] = frame.get("split", split).fillna(split)
        frame["metrics_path"] = str(path)
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(f"No metrics files found under {processed_root}")
    return pd.concat(frames, ignore_index=True)

metrics_df = load_metrics(PROCESSED_ROOT)
metrics_df.head()

In [ ]:
summary = (
    metrics_df.groupby(["dataset", "model", "split"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["dataset", "model", "split"])
)
summary

## Configure Experiment Filters

Leave `selected_datasets` or `selected_models` as `None` to include everything currently available.

In [ ]:
available_metrics = [
    "prompt_mu_score",
    "prompt_sigma_score",
    "perplexity",
    "num_tokens",
    "text_len_symbols",
    "text_len_words",
    "text_len_sentences",
]
available_metrics = [metric for metric in available_metrics if metric in metrics_df.columns]

selected_datasets = None
selected_models = None
selected_metrics = [
    "prompt_mu_score",
    "prompt_sigma_score",
    "perplexity",
    "num_tokens",
    "text_len_words",
]
selected_metrics = [metric for metric in selected_metrics if metric in available_metrics]

plot_df = metrics_df.copy()
if selected_datasets is not None:
    plot_df = plot_df[plot_df["dataset"].isin(selected_datasets)]
if selected_models is not None:
    plot_df = plot_df[plot_df["model"].isin(selected_models)]

print(f"Rows selected: {len(plot_df):,}")
print(f"Datasets: {sorted(plot_df['dataset'].unique())}")
print(f"Models: {sorted(plot_df['model'].unique())}")
print(f"Metrics: {selected_metrics}")

## Distribution Helpers

In [ ]:
def plot_metric_distributions(data: pd.DataFrame, metric: str, *, bins: int = 50) -> None:
    grid = sns.displot(
        data=data,
        x=metric,
        hue="split",
        row="dataset",
        col="model",
        kind="hist",
        bins=bins,
        stat="density",
        common_norm=False,
        element="step",
        height=3.0,
        aspect=1.25,
        facet_kws={"sharex": False, "sharey": False},
    )
    grid.set_titles(row_template="{row_name}", col_template="{col_name}")
    grid.set_axis_labels(metric, "density")
    grid.figure.suptitle(f"Distribution of {metric} by dataset/model/split", y=1.03)
    plt.show()

def plot_experiment_metric_row(data: pd.DataFrame, dataset: str, model: str, metrics: list[str]) -> None:
    experiment_df = data[(data["dataset"] == dataset) & (data["model"] == model)]
    if experiment_df.empty:
        raise ValueError(f"No rows for dataset={dataset!r}, model={model!r}")

    fig, axes = plt.subplots(1, len(metrics), figsize=(4.2 * len(metrics), 3.5), constrained_layout=True)
    if len(metrics) == 1:
        axes = [axes]
    for axis, metric in zip(axes, metrics):
        sns.histplot(
            data=experiment_df,
            x=metric,
            hue="split",
            bins=50,
            stat="density",
            common_norm=False,
            element="step",
            ax=axis,
        )
        axis.set_title(metric)
        axis.set_ylabel("density")
    fig.suptitle(f"{dataset} / {model}", y=1.08)
    plt.show()

## Compare One Metric Across All Experiments

This view is useful for quickly scanning whether HG and MG distributions separate consistently across datasets and LLMs.

In [ ]:
for metric in selected_metrics:
    plot_metric_distributions(plot_df, metric)

## Inspect Each Dataset/Model Combination

This produces one image per experiment with the selected metrics arranged in a single row.

In [ ]:
for dataset, model in plot_df[["dataset", "model"]].drop_duplicates().itertuples(index=False):
    plot_experiment_metric_row(plot_df, dataset, model, selected_metrics)

## Numeric Summary

Use this table to compare central tendency and spread alongside the plots.

In [ ]:
plot_df.groupby(["dataset", "model", "split"])[selected_metrics].describe().round(4)